# AIDA — B Cell Annotation Bias Analysis

This notebook investigates why CellTypist annotation F1 for **B cell** (generic B label) decreases monotonically as SG_Chinese proportion in the reference rises from 0 % → 100 %.

Key result: Kendall τ = −0.587, p = 0.0125; mean F1 0.499 → 0.449 (−0.050).

In [2]:
# ── Setup ──────────────────────────────────────────────────────────────────
!pip install scanpy celltypist -q
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/AIDA_project')

import scanpy as sc
import numpy as np
import pandas as pd
import json
import scipy.stats as stats
import scipy.sparse as sp
from sklearn.metrics import f1_score, classification_report
import celltypist

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 613.7 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.2/190.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.3/405.3 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 M

/usr/local/lib/python3.13/dist-packages/celltypist/classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv


In [3]:
# ── Setup ──────────────────────────────────────────────────────────────────
import scanpy as sc
import numpy as np
import pandas as pd
import json
import scipy.stats as stats
import scipy.sparse as sp
from sklearn.metrics import f1_score, classification_report
import celltypist

In [5]:
# ── Load AIDA in backed mode + fix query donors (seed=42) ──────────────────
adata = sc.read_h5ad('AIDA.h5ad', backed='r')
print(adata)

sg_chinese_donors = adata.obs[
    adata.obs['self_reported_ethnicity'] == 'Singaporean Chinese'
]['donor_id'].unique()

np.random.seed(42)
query_donors = np.random.choice(sg_chinese_donors, size=20, replace=False)
ref_candidate_donors = sg_chinese_donors[~np.isin(sg_chinese_donors, query_donors)]

all_other_donors = adata.obs[
    adata.obs['self_reported_ethnicity'].isin([
        'Japanese', 'Korean', 'Thai', 'Indian',
        'Singaporean Malay', 'Singaporean Indian'
    ])
]['donor_id'].unique()

print(f"Query donors       : {len(query_donors)}")
print(f"SG_Chinese ref pool: {len(ref_candidate_donors)}")
print(f"Other-ethnic pool  : {len(all_other_donors)}")


AnnData object with n_obs × n_vars = 1265624 × 35477 backed at 'AIDA.h5ad'
    obs: 'reference_genome', 'gene_annotation_version', 'alignment_software', 'intronic_reads_counted', 'library_id', 'assay_ontology_term_id', 'sequenced_fragment', 'cell_number_loaded', 'institute', 'is_primary_data', 'cell_type_ontology_term_id', 'author_cell_type', 'sample_id', 'sample_preservation_method', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'sample_collection_method', 'donor_BMI_at_collection', 'tissue_type', 'suspension_derivation_process', 'suspension_enriched_cell_types', 'cell_viability_percentage', 'suspension_uuid', 'suspension_type', 'donor_id', 'self_reported_ethnicity_ontology_term_id', 'donor_living_at_sample_collection', 'disease_ontology_term_id', 'sex_ontology_term_id', 'nCount_RNA', 'nFeature_RNA', 'pMito', 'NODG', 'nUMI', 'Country', 'Annotation_Level1', 'Annotation_Level2', 'Annotation_Level3', 'Annotation_Level4', 'Smoking Status', 'cell_type', 'assay', 'disease

## 1. Generic B cell count and annotation quality in the query set

`cell_type == 'B cell'` is the **generic** B label — cells not assignable to
naive B, memory B, or mature B. We inspect their AIDA Level-4 sub-labels.

In [8]:
# ── Generic B cells in query ───────────────────────────────────────────────
query_mask = adata.obs['donor_id'].isin(query_donors)
b_cells_query = adata.obs[query_mask & (adata.obs['cell_type'] == 'B cell')]

print(f"Total generic B cells in query: {len(b_cells_query)}")
print()
print(b_cells_query[['Annotation_Level1', 'Annotation_Level2',
                      'Annotation_Level3', 'Annotation_Level4']].value_counts().head(20))

Total generic B cells in query: 75

Annotation_Level1  Annotation_Level2  Annotation_Level3  Annotation_Level4      
B                  B                  B                  flagged_platelet_sum       32
                                                         B_unknown                  20
                                                         flagged_B_low_exp          13
                                                         flagged_B_TNK_gene          6
                                                         flagged_B_platelet_gene     4
                                                         CD4+_T_cm                   0
                                                         CD4+_T_cm_IFNhi             0
                                                         CD4+_T_cyt                  0
                                                         CD4+_T_em                   0
                                                         CD4+_T_memory               0
             

## 2. Generic B proportion: SG_Chinese vs. other ethnicities

We check whether SG_Chinese donors have more or fewer generic B cells than other groups.

In [9]:
# ── Generic B proportion per ethnicity group ───────────────────────────────
b_labels        = ['B cell', 'naive B cell', 'memory B cell', 'mature B cell']
other_ethnicities = ['Japanese', 'Korean', 'Thai', 'Indian',
                     'Singaporean Malay', 'Singaporean Indian']

# Per-ethnicity breakdown
ethnicities_all = ['Singaporean Chinese'] + other_ethnicities
rows = []
for eth in ethnicities_all:
    eth_b = adata.obs[
        (adata.obs['self_reported_ethnicity'] == eth) &
        (adata.obs['cell_type'].isin(b_labels))
    ]
    total   = len(eth_b)
    generic = (eth_b['cell_type'] == 'B cell').sum()
    rows.append({'ethnicity': eth, 'total_b': total,
                 'generic_b': generic,
                 'generic_b_pct': round(generic / total * 100, 1) if total > 0 else 0})

df_eth = pd.DataFrame(rows).set_index('ethnicity')
print(df_eth)

# Pooled comparison
sg_b    = adata.obs[(adata.obs['self_reported_ethnicity'] == 'Singaporean Chinese') &
                    (adata.obs['cell_type'].isin(b_labels))]
other_b = adata.obs[(adata.obs['self_reported_ethnicity'].isin(other_ethnicities)) &
                    (adata.obs['cell_type'].isin(b_labels))]

sg_gen    = (sg_b['cell_type'] == 'B cell').sum()
other_gen = (other_b['cell_type'] == 'B cell').sum()

print(f"\nSG_Chinese:   {sg_gen}/{len(sg_b)} = {sg_gen/len(sg_b)*100:.1f}% generic B")
print(f"Other ethnic: {other_gen}/{len(other_b)} = {other_gen/len(other_b)*100:.1f}% generic B")
print(f"Difference  : {sg_gen/len(sg_b)*100 - other_gen/len(other_b)*100:.1f} pp")

                     total_b  generic_b  generic_b_pct
ethnicity                                             
Singaporean Chinese     8090        387            4.8
Japanese               19757        892            4.5
Korean                 21535       2272           10.6
Thai                    6752        404            6.0
Indian                  3572       3572          100.0
Singaporean Malay       8602        457            5.3
Singaporean Indian      8915        451            5.1

SG_Chinese:   387/8090 = 4.8% generic B
Other ethnic: 8048/69133 = 11.6% generic B
Difference  : -6.9 pp


## 3. B cell composition at Annotation_Level4

We look at the full Level-4 landscape across all ethnicities. Note the **Indian** donor
anomaly: 100 % of their Annotation_Level1='B' cells are labelled `B_unknown`,
which is a data quality issue for that sub-cohort.

In [10]:
# ── Level-4 composition across ethnicities (all B cells in Level-1='B') ───
b_level4_cells = adata.obs[adata.obs['Annotation_Level1'] == 'B']

b_level4 = b_level4_cells.groupby(['self_reported_ethnicity', 'Annotation_Level4']).size()
b_total  = b_level4_cells.groupby('self_reported_ethnicity').size()
b_pct    = (b_level4 / b_total * 100).unstack(fill_value=0)

flag_cols = ['flagged_platelet_sum', 'B_unknown', 'flagged_B_low_exp',
             'flagged_B_platelet_gene', 'flagged_B_TNK_gene']
print(b_pct[[c for c in flag_cols if c in b_pct.columns]].round(1))
print()
print("NOTE: Indian donors show 100% B_unknown — likely a data quality or batch issue.")

Annotation_Level4        flagged_platelet_sum  B_unknown  flagged_B_low_exp  \
self_reported_ethnicity                                                       
Japanese                                  1.1        2.4                0.1   
Korean                                    7.6        1.2                0.1   
Singaporean Chinese                       1.7        1.3                1.0   
Indian                                    0.0      100.0                0.0   
Thai                                      3.4        0.9                0.3   
Singaporean Malay                         1.5        1.6                1.2   
Singaporean Indian                        1.8        1.6                0.8   
unknown                                   3.2        3.1                0.5   

Annotation_Level4        flagged_B_platelet_gene  flagged_B_TNK_gene  
self_reported_ethnicity                                               
Japanese                                     0.5                 0.

In [7]:
# ── Full Level-4 composition of query B cells (Annotation_Level1 = 'B') ───
query_b_full = adata.obs[
    adata.obs['donor_id'].isin(query_donors) &
    (adata.obs['Annotation_Level1'] == 'B')
]

total = len(query_b_full)
print(f"Query B cells total (Level-1 = 'B'): {total}")
print()
print((query_b_full['Annotation_Level4'].value_counts() / total * 100).round(1).head(20))

Query B cells total (Level-1 = 'B'): 1606

Annotation_Level4
naive_B                       39.2
memory_B_IGHMlo               25.3
memory_B_IGHMhi               13.7
atypical_B_FCRL5hi_FCRL3hi     4.5
memory_B_unknown               3.7
atypical_B                     3.0
naive_B_PLD4hi                 2.3
flagged_platelet_sum           2.0
B_unknown                      1.2
atypical_B_FCRL3hi             1.2
memory_B_EGR1hi                0.9
naive_B_IFNhi                  0.9
flagged_B_low_exp              0.8
memory_B_IFNhi                 0.6
flagged_B_TNK_gene             0.4
flagged_B_platelet_gene        0.2
CD4+_T_cm                      0.0
CD4+_T_cm_IFNhi                0.0
CD4+_T_cyt                     0.0
CD4+_T_em                      0.0
Name: count, dtype: float64


In [8]:
# ── Level-4 composition per ethnicity (top subtypes) ──────────────────────
for eth in ethnicities_all:
    b_eth = adata.obs[
        (adata.obs['self_reported_ethnicity'] == eth) &
        (adata.obs['Annotation_Level1'] == 'B')
    ]
    total = len(b_eth)
    print(f"\n== {eth} (n={total}) ==")
    print((b_eth['Annotation_Level4'].value_counts() / total * 100).round(1).head(10))


== Singaporean Chinese (n=8090) ==
Annotation_Level4
naive_B                       39.2
memory_B_IGHMlo               26.0
memory_B_IGHMhi               13.7
naive_B_PLD4hi                 3.0
memory_B_unknown               2.8
atypical_B_FCRL5hi_FCRL3hi     2.8
atypical_B                     2.8
flagged_platelet_sum           1.7
atypical_B_FCRL3hi             1.4
naive_B_IFNhi                  1.4
Name: count, dtype: float64

== Japanese (n=19757) ==
Annotation_Level4
naive_B                       45.8
memory_B_IGHMlo               24.0
memory_B_IGHMhi               12.6
memory_B_unknown               3.0
naive_B_PLD4hi                 2.8
B_unknown                      2.4
atypical_B                     2.0
naive_B_IFNhi                  1.8
atypical_B_FCRL5hi_FCRL3hi     1.5
flagged_platelet_sum           1.1
Name: count, dtype: float64

== Korean (n=21535) ==
Annotation_Level4
naive_B                       27.9
memory_B_IGHMlo               27.5
memory_B_IGHMhi               17.4

## 4. Generic B cell Level-4 comparison: query SG_Chinese vs. other ethnicities

Among cells labelled `cell_type == 'B cell'` (generic), what are their Level-4 labels?
This provides a transcriptomic diagnostic for *why* the reference composition matters,
noting that CellTypist itself operates on Level-1 labels only.

In [11]:
# ── Level-4 breakdown of generic B cells only ─────────────────────────────
all_sgchinese_b = adata.obs[
    (adata.obs['self_reported_ethnicity'] == 'Singaporean Chinese') &
    (adata.obs['cell_type'] == 'B cell')
]

other_generic_b = adata.obs[
    (adata.obs['self_reported_ethnicity'].isin(other_ethnicities)) &
    (adata.obs['cell_type'] == 'B cell')
]

sgc_counts   = all_sgchinese_b['Annotation_Level4'].value_counts()
other_counts = other_generic_b['Annotation_Level4'].value_counts()

all_cats = sgc_counts[sgc_counts > 0].index.union(other_counts[other_counts > 0].index)
comparison = pd.DataFrame({
    'SG_Chinese (%)':        (sgc_counts[all_cats] / len(all_sgchinese_b) * 100).round(1),
    'Other ethnicities (%)': (other_counts[all_cats] / len(other_generic_b) * 100).round(1)
}).fillna(0).sort_values('SG_Chinese (%)', ascending=False)

print(f"SG_Chinese generic B total : {len(all_sgchinese_b)}")
print(f"Other generic B total      : {len(other_generic_b)}")
print()
print(comparison)
print()
print("Key contrast (Level-4 diagnostic; CellTypist uses Level-1 only):\n"
"  SG_Chinese: 35.4% flagged_platelet_sum, 26.4% B_unknown, 21.4% flagged_B_low_exp\n"
"  Other ethnic: 29.4% flagged_platelet_sum, 57.6% B_unknown, 2.9% flagged_B_low_exp\n"
"  → Other ethnicities have 2.4× more generic B cells overall (11.6% vs 4.8%).")

SG_Chinese generic B total : 387
Other generic B total      : 8048

                         SG_Chinese (%)  Other ethnicities (%)
Annotation_Level4                                             
flagged_platelet_sum               35.4                   29.4
B_unknown                          26.4                   57.6
flagged_B_low_exp                  21.4                    2.9
flagged_B_platelet_gene             9.3                    6.8
flagged_B_TNK_gene                  7.5                    3.3

Key contrast (Level-4 diagnostic; CellTypist uses Level-1 only):
  SG_Chinese: 35.4% flagged_platelet_sum, 26.4% B_unknown, 21.4% flagged_B_low_exp
  Other ethnic: 29.4% flagged_platelet_sum, 57.6% B_unknown, 2.9% flagged_B_low_exp
  → Other ethnicities have 2.4× more generic B cells overall (11.6% vs 4.8%).


## 5. F1 / precision / recall across SG_Chinese proportions

Results from `all_reports.json` (36 replicates × 11 proportions).

In [10]:
# ── Load results ───────────────────────────────────────────────────────────
with open('all_reports.json', 'r') as f:
    all_reports = json.load(f)

rows = []
for entry in all_reports:
    if 'B cell' in entry['report']:
        r = entry['report']['B cell']
        rows.append({
            'proportion': entry['proportion'],
            'repeat':     entry['repeat'],
            'precision':  r['precision'],
            'recall':     r['recall'],
            'f1':         r['f1-score']
        })

df_b = pd.DataFrame(rows)
summary = df_b.groupby('proportion')[['precision', 'recall', 'f1']].agg(['mean', 'std']).round(3)
print(summary)

           precision        recall            f1       
                mean    std   mean    std   mean    std
proportion                                             
0.0            0.595  0.057  0.433  0.037  0.499  0.034
0.1            0.628  0.040  0.410  0.029  0.495  0.025
0.2            0.660  0.072  0.414  0.048  0.505  0.041
0.3            0.611  0.059  0.418  0.031  0.495  0.034
0.4            0.630  0.052  0.423  0.042  0.506  0.043
0.5            0.646  0.039  0.424  0.049  0.510  0.039
0.6            0.609  0.064  0.392  0.029  0.476  0.036
0.7            0.591  0.044  0.399  0.044  0.474  0.037
0.8            0.562  0.037  0.410  0.044  0.472  0.033
0.9            0.567  0.035  0.387  0.034  0.459  0.029
1.0            0.543  0.030  0.384  0.029  0.449  0.025


In [11]:
# ── Monotonicity test (Kendall τ) ──────────────────────────────────────────
props  = summary.index.tolist()
f1vals = summary['f1']['mean'].tolist()

tau, p = stats.kendalltau(props, f1vals)
print(f"Kendall τ (B cell F1 vs. proportion): {tau:.3f}, p = {p:.4f}")
print(f"Mean F1 range: {min(f1vals):.3f} → {max(f1vals):.3f}")
print(f"Total change (100% vs 0%): {f1vals[-1] - f1vals[0]:+.3f}")

Kendall τ (B cell F1 vs. proportion): -0.587, p = 0.0125
Mean F1 range: 0.449 → 0.510
Total change (100% vs 0%): -0.050


## 6. Misclassification analysis (10 runs, 0% vs. 100% reference)

> **Note:** this cell re-runs CellTypist (10 × 2 = 20 jobs). Skip if results are
> already saved.

In [14]:
def run_celltypist(adata, query_donors, other_donors, ref_candidate_donors,
                   sg_chinese_proportion, n_ref_total=60):
    n_chinese = int(n_ref_total * sg_chinese_proportion)
    n_others  = n_ref_total - n_chinese

    chinese_ref = np.random.choice(ref_candidate_donors, size=n_chinese, replace=False) \
                  if n_chinese > 0 else []
    other_ref   = np.random.choice(other_donors, size=n_others, replace=False)
    all_ref_donors = list(chinese_ref) + list(other_ref)

    ref_adata = adata[adata.obs['donor_id'].isin(all_ref_donors)].to_memory()
    sc.pp.highly_variable_genes(ref_adata, n_top_genes=2000)
    ref_adata = ref_adata[:, ref_adata.var['highly_variable']]

    model = celltypist.train(ref_adata, labels='cell_type', n_jobs=-1,
                             with_mean=False, check_expression=False)

    query = adata[adata.obs['donor_id'].isin(query_donors)].to_memory()
    query = query[:, ref_adata.var_names].copy()
    if sp.issparse(query.X):
        query.X = query.X.toarray()

    predictions  = celltypist.annotate(query, model=model)
    true_labels  = query.obs['cell_type']
    pred_labels  = predictions.predicted_labels['predicted_labels']
    f1 = f1_score(true_labels, pred_labels, average='macro')
    return f1, predictions, query

In [15]:
# ── Misclassification accumulation over 10 runs ───────────────────────────
n_runs = 10
b_misclass = {'0%': {}, '100%': {}}

for i in range(n_runs):
    print(f"Run {i+1}/{n_runs} ...")
    for prop, key in [(0.0, '0%'), (1.0, '100%')]:
        f1, pred, query = run_celltypist(
            adata, query_donors, all_other_donors,
            ref_candidate_donors, sg_chinese_proportion=prop)
        true_lab = query.obs['cell_type']
        pred_lab = pred.predicted_labels['predicted_labels']
        b_mask   = true_lab == 'B cell'
        for label, count in pred_lab[b_mask].value_counts().items():
            b_misclass[key][label] = b_misclass[key].get(label, 0) + count

df_b_0   = pd.Series(b_misclass['0%']).sort_values(ascending=False)
df_b_100 = pd.Series(b_misclass['100%']).sort_values(ascending=False)

print("\n0% SG_Chinese reference — B cell predicted as (10-run total):")
print(df_b_0[df_b_0 > 0])

print("\n100% SG_Chinese reference — B cell predicted as (10-run total):")
print(df_b_100[df_b_100 > 0])

Run 1/10 ...


Run 2/10 ...


Run 3/10 ...


Run 4/10 ...


Run 5/10 ...


...

Run 7/10 ...


Run 8/10 ...


Run 9/10 ...


Run 10/10 ...



0% SG_Chinese reference — B cell predicted as (10-run total):
B cell                                                  316
naive B cell                                            215
memory B cell                                           175
mature B cell                                            33
platelet                                                  3
myeloid cell                                              2
naive thymus-derived CD4-positive, alpha-beta T cell      2
CD4-positive, alpha-beta T cell                           1
innate lymphoid cell                                      1
natural killer cell                                       1
regulatory T cell                                         1
dtype: int64

100% SG_Chinese reference — B cell predicted as (10-run total):
B cell                             284
naive B cell                       211
memory B cell                      191
mature B cell                       45
T cell                               9
natu

## Summary

| | 0 % SG_Chinese ref | 100 % SG_Chinese ref |
|---|---|---|
| Mean F1 | 0.499 | 0.449 |
| Mean precision | 0.595 | 0.543 |
| Mean recall | 0.433 | 0.384 |
| Kendall τ | −0.587 | p = 0.0125 |

**Mechanism:** CellTypist trains only on Level-1 labels ("B cell") and does not see
Level-4 annotations. The F1 decrease with more SG_Chinese reference reflects a
training example scarcity effect: SG_Chinese donors have only 4.8 % generic B cells,
versus 11.6 % in other-ethnicity donors (a 2.4× difference). Shifting the reference
toward SG_Chinese therefore reduces the absolute number of "B cell" training examples,
weakening recall. Level-4 analysis (diagnostic): other-ethnicity generic B cells are
57.6 % B_unknown vs. 26.4 % for SG_Chinese, suggesting additional transcriptomic
heterogeneity in other-ethnicity exemplars.
